In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import tempfile
import urllib.request

import librosa

# Add the parent directory to the path so we can import from src
sys.path.insert(0, "..")

from src.melt.processing_melt import MELT_SPECIAL_TOKENS, MELTProcessor
from transformers import AutoFeatureExtractor, AutoTokenizer

## Setup: Load Components and Audio Sample

In [3]:
# Audio sample URL for testing
AUDIO_SAMPLE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"
AUDIO_SAMPLE_URL_2 = "recording.flac"

# Download and load audio sample
def load_audio_sample(url):
    """Load audio sample from URL using librosa."""
    with urllib.request.urlopen(url) as response:
        audio_bytes = response.read()
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_path = tmp_file.name
    
    audio, sr = librosa.load(tmp_path, sr=16000)
    return audio

def load_audio_local(path):
    """Load audio sample from local path using librosa."""
    audio, sr = librosa.load(path, sr=16000)
    return audio

audio_sample = load_audio_sample(AUDIO_SAMPLE_URL)
print(f"Audio sample shape: {audio_sample.shape}")
print(f"Audio duration: {len(audio_sample) / 16000:.2f} seconds")
audio_sample_2 = load_audio_local(AUDIO_SAMPLE_URL_2)
print(f"Audio sample 2 shape: {audio_sample_2.shape}")
print(f"Audio 2 duration: {len(audio_sample_2) / 16000:.2f} seconds")

Audio sample shape: (144000,)
Audio duration: 9.00 seconds
Audio sample 2 shape: (364324,)
Audio 2 duration: 22.77 seconds


In [68]:
# Load feature extractor and tokenizer
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

# Create processor
processor = MELTProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Processor created successfully!")
print(f"Audio token: {processor.audio_token}")
print(f"Audio BOS token: {processor.audio_bos_token}")
print(f"Audio EOS token: {processor.audio_eos_token}")

Processor created successfully!
Audio token: <|AUDIO|>
Audio BOS token: <|audio_bos|>
Audio EOS token: <|audio_eos|>


In [69]:
def display_result(result, processor, title="Result"):
    """Helper function to display processor output."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    print(f"\nKeys in result: {list(result.keys())}")
    
    # Handle both list and tensor inputs
    input_ids = result["input_ids"]
    if hasattr(input_ids, "shape"):
        print(f"\ninput_ids shape: {input_ids.shape}")
        num_samples = input_ids.shape[0]
    else:
        print(f"\ninput_ids: list of {len(input_ids)} samples")
        num_samples = len(input_ids)
    
    if "input_features" in result:
        features = result["input_features"]
        if hasattr(features, "shape"):
            print(f"input_features shape: {features.shape}")
        else:
            print(f"input_features: list of {len(features)} items")
    
    print(f"\n--- Decoded text for each sample ---")
    for i in range(num_samples):
        if hasattr(input_ids, "shape"):
            ids = input_ids[i]
        else:
            ids = input_ids[i]
        decoded = processor.decode(ids)
        print(f"\nSample {i+1}:")
        print(f"  Token count: {len(ids)}")
        print(f"  Decoded (first 500 chars):")
        print(f"  {decoded[:500]}..." if len(decoded) > 500 else f"  {decoded}")

    
    print(list(result.keys()))

    if isinstance(result["input_ids"], list):
        print("input_ids:", result["input_ids"])
    else:
        print("input_ids:", result["input_ids"].shape)

    if result.get("input_features") is not None:
        print("input_features:", result["input_features"].shape)
        print("features_attention_mask:", result["features_attention_mask"].shape)
        print("audio_lengths:", result["audio_lengths"])

## 1. Text Only Processing

### 1.1 Simple Text (No Chat Template)

In [111]:
text = "Hello, how are you today?"
result = processor(text=text)

display_result(result, processor, "Simple Text (No Chat Template)")


Simple Text (No Chat Template)

Keys in result: ['input_ids', 'attention_mask']

input_ids: list of 1 samples

--- Decoded text for each sample ---

Sample 1:
  Token count: 7
  Decoded (first 500 chars):
  Hello, how are you today?
['input_ids', 'attention_mask']
input_ids: [[9707, 11, 1246, 525, 498, 3351, 30]]


### 1.2 Text with Chat Template

In [112]:
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing well, thank you!"},
]

# Apply chat template
text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

result = processor(text=text_with_template)
display_result(result, processor, "Text with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you!<|im_end|>
<|im_start|>assistant



Text with Chat Template

Keys in result: ['input_ids', 'attention_mask']

input_ids: list of 1 samples

--- Decoded text for each sample ---

Sample 1:
  Token count: 38
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you!<|im_end|>
<|im_start|>assistant

['input_ids', 'attention_mask']
input_ids: [[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 9707, 11, 1246, 525, 498, 30, 151645, 198, 151644, 77091, 198, 40, 2776, 3730, 1632, 11, 9702, 498, 0, 151645, 198, 151644, 77091, 198]]


## 2. Single Audio Processing

### 2.1 Single Audio (No Chat Template)

In [113]:
audio_token = processor.audio_token
text = f"Transcribe the following audio: {audio_token}"

print(f"Input text: {text}")
print()

result = processor(text=text, audio=audio_sample, return_tensors="pt")
display_result(result, processor, "Single Audio (No Chat Template)")

Input text: Transcribe the following audio: <|AUDIO|>


Single Audio (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([1, 458])
input_features shape: torch.Size([1, 452, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 458
  Decoded (first 500 chars):
  Transcribe the following audio: <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUD...
['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']
input_ids: torch.

### 2.2 Single Audio with Chat Template

In [114]:
audio_token = processor.audio_token
messages = [
    {"role": "user", "content": f"Transcribe the following audio: {audio_token}"},
]

text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

result = processor(text=text_with_template, audio=audio_sample, return_tensors="pt")
display_result(result, processor, "Single Audio with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Transcribe the following audio: <|AUDIO|><|im_end|>
<|im_start|>assistant



Single Audio with Chat Template

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([1, 477])
input_features shape: torch.Size([1, 452, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 477
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Transcribe the following audio: <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|>

## 3. Multiple Audios Per Sample

### 3.1 Three Audios (No Chat Template)

In [115]:
audio_token = processor.audio_token
text = f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all."
audios = [audio_sample, audio_sample_2, audio_sample]

print(f"Input text: {text}")
print(f"Number of audios: {len(audios)}")
print()

result = processor(text=text, audio=audios, return_tensors="pt")
display_result(result, processor, "Three Audios (No Chat Template)")


Input text: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Number of audios: 3


Three Audios (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([1, 2058])
input_features shape: torch.Size([1, 2044, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 2058
  Decoded (first 500 chars):
  <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><...
['input_ids', 'attention_mask', 'input_features',

### 3.2 Three Audios with Chat Template

In [116]:
audio_token = processor.audio_token
messages = [
    {
        "role": "user",
        "content": f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all.",
    },
]

text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

audios = [audio_sample, audio_sample, audio_sample]
result = processor(text=text_with_template, audio=audios, return_tensors="pt")
display_result(result, processor, "Three Audios with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.<|im_end|>
<|im_start|>assistant



Three Audios with Chat Template

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([1, 1389])
input_features shape: torch.Size([1, 1356, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 1389
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|

### 3.3 Two Audios for Comparison

In [117]:
audio_token = processor.audio_token
text = f"Compare {audio_token} with {audio_token}"
audios = [audio_sample, audio_sample_2]

print(f"Input text: {text}")
print(f"Number of audios: {len(audios)}")
print()

result = processor(text=text, audio=audios, return_tensors="pt")
display_result(result, processor, "Two Audios for Comparison")

Input text: Compare <|AUDIO|> with <|AUDIO|>
Number of audios: 2


Two Audios for Comparison

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([1, 1594])
input_features shape: torch.Size([1, 1592, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 1594
  Decoded (first 500 chars):
  Compare <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|...
['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']
input_ids

## 4. Batch Processing with Different Audio Counts

### 4.1 Batch: Sample 1 (3 audios) + Sample 2 (1 audio)

In [118]:
audio_token = processor.audio_token

# Sample 1 has 3 audios
text1 = f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all."
# Sample 2 has 1 audio
text2 = f"Transcribe: {audio_token}"

texts = [text1, text2]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios = [
    [audio_sample, audio_sample_2, audio_sample],
    [audio_sample]
]

print("Sample 1 text:", text1)
print("Sample 2 text:", text2)
print(f"Total audios: {len(audios)}")
print()

result = processor(text=texts, audio=audios, return_tensors="pt", padding=True)
display_result(result, processor, "Batch: 3 audios + 1 audio")

Sample 1 text: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Sample 2 text: Transcribe: <|AUDIO|>
Total audios: 2


Batch: 3 audios + 1 audio

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([2, 2058])
input_features shape: torch.Size([2, 2044, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 2058
  Decoded (first 500 chars):
  <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><...

Sample 2:
  Token 

### 4.2 Batch: Sample 1 (1 audio) + Sample 2 (2 audios)

In [119]:
audio_token = processor.audio_token

# Sample 1 has 1 audio
text1 = f"Single audio sample: {audio_token}"
# Sample 2 has 2 audios
text2 = f"Compare {audio_token} with {audio_token}"

texts = [text1, text2]
# 1 audio for sample 1, 2 audios for sample 2 = 3 total
audios = [[audio_sample], [audio_sample, audio_sample]]


print("Sample 1 text:", text1)
print("Sample 2 text:", text2)
print(f"Total audios: {sum(len(a) for a in audios)}")
print()

result = processor(text=texts, audio=audios, return_tensors="pt", padding=True)
display_result(result, processor, "Batch: 1 audio + 2 audios")

Sample 1 text: Single audio sample: <|AUDIO|>
Sample 2 text: Compare <|AUDIO|> with <|AUDIO|>
Total audios: 3


Batch: 1 audio + 2 audios

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids shape: torch.Size([2, 906])
input_features shape: torch.Size([2, 904, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 906
  Decoded (first 500 chars):
  <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endo...

Sample 2:
  Token count: 906
  Decoded (first 500 chars):
  

### 4.3 Batch with Chat Template: Mixed Audio Counts

In [121]:
audio_token = processor.audio_token

# Sample 1: multi-audio with chat template
messages1 = [
    {
        "role": "user",
        "content": f"{audio_token} Describe this. {audio_token} And this.",
    },
]
text1 = processor.tokenizer.apply_chat_template(
    messages1, tokenize=False, add_generation_prompt=True
)

# Sample 2: single audio with chat template
messages2 = [
    {"role": "user", "content": f"What do you hear? {audio_token}"},
]
text2 = processor.tokenizer.apply_chat_template(
    messages2, tokenize=False, add_generation_prompt=True
)

texts = [text1, text2]
# 2 audios for sample 1, 1 audio for sample 2 = 3 total
audios = [[audio_sample, audio_sample], [audio_sample]]

print("Sample 1 (chat template applied):")
print(text1)
print("\nSample 2 (chat template applied):")
print(text2)
print(f"\nTotal audios: {len(audios)}")
print()

result = processor(text=texts, audio=audios, padding=True)
display_result(result, processor, "Batch with Chat Template: 2 audios + 1 audio")

Sample 1 (chat template applied):
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|AUDIO|> Describe this. <|AUDIO|> And this.<|im_end|>
<|im_start|>assistant


Sample 2 (chat template applied):
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What do you hear? <|AUDIO|><|im_end|>
<|im_start|>assistant


Total audios: 2


Batch with Chat Template: 2 audios + 1 audio

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask', 'audio_lengths']

input_ids: list of 2 samples
input_features shape: torch.Size([2, 904, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 928
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AU

## 5. Examining Audio Token Expansion

In [122]:
audio_token = processor.audio_token
text = f"Before {audio_token} after"

print(f"Input text: {text}")
print()

result = processor(text=text, audio=audio_sample)

# Count occurrences of audio token in decoded output
decoded = processor.decode(result["input_ids"][0])
audio_token_count = decoded.count(audio_token)

print(f"Audio token count in decoded output: {audio_token_count}")
print(f"\nFull decoded output:")
print(decoded)

Input text: Before <|AUDIO|> after

Audio token count in decoded output: 449

Full decoded output:
Before <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|

## 6. Summary: Special Tokens Available

In [123]:
print("MELT Special Tokens:")
print("=" * 40)
for name, token in MELT_SPECIAL_TOKENS.items():
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f"{name:20s}: {token:15s} (ID: {token_id})")

MELT Special Tokens:
image_token         : <|IMAGE|>       (ID: 151665)
audio_token         : <|AUDIO|>       (ID: 151666)
video_token         : <|VIDEO|>       (ID: 151667)
vision_bos_token    : <|vision_bos|>  (ID: 151668)
vision_eos_token    : <|vision_eos|>  (ID: 151669)
audio_bos_token     : <|audio_bos|>   (ID: 151670)
audio_eos_token     : <|audio_eos|>   (ID: 151671)


## 7. Comparison with Qwen2AudioProcessor and Phi4MultimodalProcessor

Let's compare how MELTProcessor handles batched inputs with different audio counts against the official Qwen2AudioProcessor and Phi4MultimodalProcessor implementations.

### 7.1 Qwen2AudioProcessor Comparison

In [17]:
# Load Qwen2AudioProcessor
from transformers import Qwen2AudioProcessor

qwen2_processor = Qwen2AudioProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
print("Qwen2AudioProcessor loaded!")
print(f"Audio token: {qwen2_processor.audio_token}")
print(f"Audio BOS token: {qwen2_processor.audio_bos_token}")
print(f"Audio EOS token: {qwen2_processor.audio_eos_token}")

Qwen2AudioProcessor loaded!
Audio token: <|AUDIO|>
Audio BOS token: <|audio_bos|>
Audio EOS token: <|audio_eos|>


In [18]:
# Qwen2Audio: Batch with 3 audios in sample 1, 1 audio in sample 2
qwen_audio_token = qwen2_processor.audio_token

# Sample 1: 3 audios
text1_qwen = f"{qwen_audio_token} What is said here? {qwen_audio_token} And in this one? {qwen_audio_token} Summarize all."
# Sample 2: 1 audio  
text2_qwen = f"Transcribe: {qwen_audio_token}"

texts_qwen = [text1_qwen, text2_qwen]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios_qwen = [audio_sample, audio_sample, audio_sample, audio_sample]

print("Qwen2Audio Input:")
print(f"Sample 1: {text1_qwen}")
print(f"Sample 2: {text2_qwen}")
print(f"Total audios: {len(audios_qwen)}")
print()

try:
    result_qwen = qwen2_processor(text=texts_qwen, audio=audios_qwen, padding=True)
    display_result(result_qwen, qwen2_processor, "Qwen2AudioProcessor: Batch 3+1 audios")
except Exception as e:
    print(f"Error with Qwen2AudioProcessor: {e}")

It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


Qwen2Audio Input:
Sample 1: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Sample 2: Transcribe: <|AUDIO|>
Total audios: 4


Qwen2AudioProcessor: Batch 3+1 audios

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'features_attention_mask']

input_ids: list of 2 samples
input_features shape: (4, 128, 3000)

--- Decoded text for each sample ---

Sample 1:
  Token count: 698
  Decoded (first 500 chars):
  <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><...

Sample 2:
  Token count: 698
  Decoded